# Dataset Raw de xG - EDA

> **Múltiples ligas**&nbsp;&nbsp;◦&nbsp;&nbsp;**Temporadas:** 2014-15 → 2023-24&nbsp;&nbsp;◦&nbsp;&nbsp;**Fuente:** Understat (vía soccerdata)

## Objetivos

- Verificar cobertura temporal y por liga (2014-15 – 2023-24, 3 ligas).
- Identificar variables disponibles y tipos de datos.
- Validar integridad: duplicados, nulos, negativos, drift y outliers de xG.
- Comparar nomenclatura de equipos, ligas, temporadas y fechas con el core clean.
- Exportar dataset xG raw en Parquet.

## Estructura del Notebook

| # | Sección | Objetivo |
|---|---------|----------|
| 0 | Entorno y configuración | Librerías, rutas y datasets de referencia |
| 1 | Extracción de datos | Descarga de xG desde Understat vía soccerdata |
| 2 | Inspección inicial | Dimensiones, índice y tipos de dato |
| 3 | Cobertura temporal | Partidos por liga y temporada |
| 4 | Validaciones de integridad | Duplicados, nulos, negativos y outliers |
| 5 | Compatibilidad con football-data | Equipos, fechas, ligas y temporadas |
| 6 | Conclusiones | Hallazgos y decisiones para el merge |
| 7 | Exportación | Guardado del dataset xG en Parquet |


---
##

## 0) Entorno y configuración

Configuración de dependencias, rutas del proyecto y parámetros de extracción.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import soccerdata as sd
import sys
import json
from IPython.display import display, Markdown

# Rutas del proyecto
PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_ROOT = PROJECT_ROOT / "config"
RAW_XG_ROOT = PROJECT_ROOT / "data" / "raw" / "xg"
XG_RAW_PATH = RAW_XG_ROOT / "xg_validated.parquet"
XG_VALIDATED_SCHEMA_PATH = RAW_XG_ROOT / "xg_validated_schema.json"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
CORE_CLEAN_PATH = PROCESSED_ROOT / "core_multi_league_clean.parquet"

# Importación de funciones propias
from src.analysis import build_team_mapping
from src.cleaning import DataValidator


[03/22/26 20:00:10] INFO     No custom team name replacements found. You can configure these in       ]8;id=764389;file:///Users/jorgepais/Desktop/kraken/football-analytics/.venv/lib/python3.13/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=839688;file:///Users/jorgepais/Desktop/kraken/football-analytics/.venv/lib/python3.13/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/jorgepais/soccerdata/config/teamname_replacements.json.                        

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=453546;file:///Users/jorgepais/Desktop/kraken/football-analytics/.venv/lib/python3.13/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=818238;file:///Users/jorgepais/Desktop/kraken/football-analytics/.venv/lib/python3.13/site-packages/soccerdata/_config.py#198\198]8;;\
                             /Users/jorgepais/soccerdata/config/league_dict.json.                                  

### 0.1 Configuración de referencia

In [2]:
with open(CONFIG_ROOT / "leagues.json") as f:
    ALL_LEAGUES = json.load(f)

with open(PROCESSED_ROOT / "core_multi_league_clean_schema.json") as f:
    schema = json.load(f)

with open(CONFIG_ROOT / "league_mapping.json") as f:
    LEAGUE_MAP = json.load(f)

df_core = pd.read_parquet(CORE_CLEAN_PATH)
print(f"Core clean cargado: {len(df_core):,} partidos × {len(df_core.columns)} columnas")

Core clean cargado: 10,660 partidos × 33 columnas


---
##

## 1) Extracción de datos

Descarga de estadísticas de xG a nivel de partido desde `Understat` vía `soccerdata`.

In [3]:
SEASONS = [f"{y}{y+1}" for y in range(14, 24)]
# → ['1415', '1516', ..., '2324'] --> Formato interno de Soccerdata

us = sd.Understat(leagues=list(LEAGUE_MAP.keys()), seasons=SEASONS)
df_xg = us.read_schedule()

print(f"Dataset xG extraído: {df_xg.shape[0]:,} partidos × {df_xg.shape[1]} variables")
print(f"Temporadas: 20{SEASONS[0][:2]}-20{SEASONS[0][2:]} – 20{SEASONS[-1][:2]}-20{SEASONS[-1][2:]}")


                    INFO     Saving cached data to /Users/jorgepais/soccerdata/data/Understat        ]8;id=597631;file:///Users/jorgepais/Desktop/kraken/football-analytics/.venv/lib/python3.13/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=203515;file:///Users/jorgepais/Desktop/kraken/football-analytics/.venv/lib/python3.13/site-packages/soccerdata/_common.py#249\249]8;;\

[2026-03-22 20:00:10] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: /Users/jorgepais/Desktop/kraken/football-analytics/.venv/lib/python3.13/site-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dylib


                    INFO     Successfully loaded TLS library:                                      ]8;id=611800;file:///Users/jorgepais/Desktop/kraken/football-analytics/.venv/lib/python3.13/site-packages/tls_requests/models/libraries.py\libraries.py]8;;\:]8;id=26426;file:///Users/jorgepais/Desktop/kraken/football-analytics/.venv/lib/python3.13/site-packages/tls_requests/models/libraries.py#397\397]8;;\
                             /Users/jorgepais/Desktop/kraken/football-analytics/.venv/lib/python3.                 
                             13/site-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dyli                 
                             b                                                                                     

                    WARNING  /Users/jorgepais/Desktop/kraken/football-analytics/.venv/lib/python3.1 ]8;id=589790;file:///opt/homebrew/Cellar/python@3.13/3.13.12_1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/warnings.py\warnings.py]8;;\:]8;id=869684;file:///opt/homebrew/Cellar/python@3.13/3.13.12_1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/warnings.py#110\110]8;;\
                             3/site-packages/soccerdata/_common.py:143: UserWarning: Season id                     
                             "2021" is ambiguous: interpreting as "20-21"                                          
                               warnings.warn(msg, stacklevel=1)                                                    
                                                                                                                   

Dataset xG extraído: 10,660 partidos × 17 variables
Temporadas: 2014-2015 – 2023-2024


---
##

## 2) Inspección inicial

Dimensiones, tipos de datos y vista de las primeras filas.

### 2.1 Resumen estructural del dataset

In [4]:
print(f"Dimensiones: {df_xg.shape[0]:,} partidos × {df_xg.shape[1]} variables")
print(f"Índice: {df_xg.index.names} ({df_xg.index.nlevels} niveles)")
print(f"Ligas: {df_xg.index.get_level_values('league').unique().tolist()}")
print(f"Temporadas: {df_xg.index.get_level_values('season').nunique()}")

Dimensiones: 10,660 partidos × 17 variables
Índice: ['league', 'season', 'game'] (3 niveles)
Ligas: ['ENG-Premier League', 'ESP-La Liga', 'GER-Bundesliga']
Temporadas: 10


### 2.2 Tipos de datos

In [5]:
tipos = pd.DataFrame({"variable": df_xg.dtypes.index, "tipo": df_xg.dtypes.values})
display(tipos.style.hide(axis="index").set_table_attributes('style="max-height:300px; overflow-y:auto; display:block"'))


variable,tipo
league_id,string
season_id,Int64
game_id,Int64
date,datetime64[ns]
home_team_id,Int64
away_team_id,Int64
home_team,string
away_team,string
away_team_code,string
home_team_code,string


### 2.3 Preview del dataset

In [6]:
display(df_xg.head(5).style.format({"home_xg": "{:.2f}", "away_xg": "{:.2f}"}).hide(axis="index"))

league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,home_goals,away_goals,home_xg,away_xg,is_result,has_data,url
1,2014,4755,2014-08-16 17:30:00,83,78,Arsenal,Crystal Palace,CRY,ARS,2,1,1.55,0.16,True,True,https://understat.com/match/4755
1,2014,4750,2014-08-16 15:00:00,75,72,Leicester,Everton,EVE,LEI,2,2,1.28,0.61,True,True,https://understat.com/match/4750
1,2014,4749,2014-08-16 12:45:00,89,84,Manchester United,Swansea,SWA,MUN,1,2,1.17,0.28,True,True,https://understat.com/match/4749
1,2014,4751,2014-08-16 15:00:00,202,91,Queens Park Rangers,Hull,HUL,QPR,0,1,1.90,1.12,True,True,https://understat.com/match/4751
1,2014,4752,2014-08-16 15:00:00,85,71,Stoke,Aston Villa,AVL,STO,0,1,0.42,0.91,True,True,https://understat.com/match/4752


---
##

## 3) Cobertura temporal

Verificación del número de partidos por liga y temporada frente a los valores esperados del core clean.

In [7]:
n_seasons = df_xg.index.get_level_values("season").nunique()
expected = {us: schema["matches_per_league"][core] // n_seasons for us, core in LEAGUE_MAP.items()}

coverage = df_xg.groupby(level=["league", "season"]).size().unstack(fill_value=0)
coverage.columns = [f"{s[:2]}/{s[2:]}" for s in coverage.columns]
coverage.index.name = None
display(coverage)

for lg, exp in expected.items():
    gaps = coverage.loc[lg][coverage.loc[lg] != exp]
    status = "⚠" if len(gaps) else "✓"
    print(f"{status} {lg:<20} {exp}/temporada" + (f" — gaps: {gaps.to_dict()}" if len(gaps) else ""))

,14/15,15/16,16/17,17/18,18/19,19/20,20/21,21/22,22/23,23/24
ENG-Premier League,380,380,380,380,380,380,380,380,380,380
ESP-La Liga,380,380,380,380,380,380,380,380,380,380
GER-Bundesliga,306,306,306,306,306,306,306,306,306,306


✓ ENG-Premier League   380/temporada
✓ ESP-La Liga          380/temporada
✓ GER-Bundesliga       306/temporada


---
##

## 4) Validaciones de integridad

Verificación de integridad del dataset: duplicados, nulos, drift de tipos entre temporadas, valores negativos, outliers de xG, partidos sin resultado y validación cruzada de goles contra el core clean.

In [8]:
print("── Resumen integridad ─────────────────────────────────────────")
resumen = {
    "Duplicados": DataValidator.get_duplicates_count(df_xg),
    "Negativos": DataValidator.get_negatives_dict(df_xg),
    "Nulos": DataValidator.get_nulls_dict(df_xg),
    "Drift de tipos": DataValidator.get_drift_columns(df_xg, "season"),
    "Sin resultado": DataValidator.get_false_conditions(df_xg, "is_result"),
    "xG > 10": DataValidator.get_outliers_count(df_xg, ["home_xg", "away_xg"], 10.0),
}
for metrica, valor in resumen.items():
    v = valor if valor else "ninguno"
    print(f"  {metrica:<18} {v}")
print("\n ── Goles xG vs core ──────────────────────────────────────")
cruzados = DataValidator.get_goals_cross_check(df_xg, df_core, LEAGUE_MAP)
for lg_us, (match, g_xg, g_core) in cruzados.items():
    print(f"  [{match}] {lg_us:<25} xG={g_xg:>6,}  core={g_core:>6,}")
print("───────────────────────────────────────────────────────")

── Resumen integridad ─────────────────────────────────────────
  Duplicados         ninguno
  Negativos          ninguno
  Nulos              ninguno
  Drift de tipos     ninguno
  Sin resultado      ninguno
  xG > 10            ninguno

 ── Goles xG vs core ──────────────────────────────────────
  [OK] ENG-Premier League        xG=10,614  core=10,614
  [OK] ESP-La Liga               xG= 9,983  core= 9,983
  [OK] GER-Bundesliga            xG= 9,234  core= 9,234
───────────────────────────────────────────────────────


---
##

## 5) Compatibilidad con `football-data`

Comparación de nombres de equipos, ligas, temporadas y formato de fechas entre `Understat` y el core clean. 

### 5.1 Nombres de equipos

Equipos únicos por liga en cada fuente y diferencias de nomenclatura.

In [9]:
rows = []
for lg_us, lg_core in LEAGUE_MAP.items():
    teams_xg   = set(df_xg.loc[lg_us]["home_team"].unique())
    teams_core = set(df_core[df_core["League"] == lg_core]["HomeTeam"].unique())
    only_xg    = sorted(teams_xg - teams_core)
    only_core  = sorted(teams_core - teams_xg)
    
    print(f"\n── {lg_us}")
    print(f"  xG={len(teams_xg)}  core={len(teams_core)}  coinciden={len(teams_xg & teams_core)}")
    if only_xg:
        print(f"  solo xG   → {only_xg}")
    if only_core:
        print(f"  solo core → {only_core}")


── ENG-Premier League
  xG=34  core=34  coinciden=27
  solo xG   → ['Manchester City', 'Manchester United', 'Newcastle United', 'Nottingham Forest', 'Queens Park Rangers', 'West Bromwich Albion', 'Wolverhampton Wanderers']
  solo core → ['Man City', 'Man United', 'Newcastle', "Nott'm Forest", 'QPR', 'West Brom', 'Wolves']

── ESP-La Liga
  xG=31  core=31  coinciden=20
  solo xG   → ['Athletic Club', 'Atletico Madrid', 'Celta Vigo', 'Deportivo La Coruna', 'Espanyol', 'Rayo Vallecano', 'Real Betis', 'Real Sociedad', 'Real Valladolid', 'SD Huesca', 'Sporting Gijon']
  solo core → ['Ath Bilbao', 'Ath Madrid', 'Betis', 'Celta', 'Espanol', 'Huesca', 'La Coruna', 'Sociedad', 'Sp Gijon', 'Valladolid', 'Vallecano']

── GER-Bundesliga
  xG=28  core=28  coinciden=12
  solo xG   → ['Arminia Bielefeld', 'Bayer Leverkusen', 'Borussia Dortmund', 'Borussia M.Gladbach', 'Eintracht Frankfurt', 'FC Cologne', 'FC Heidenheim', 'Fortuna Duesseldorf', 'Greuther Fuerth', 'Hamburger SV', 'Hannover 96', 'Herth

#### 5.1.1 Generación del mapping de equipos

Mapping automático de nombres de equipos entre `Understat` y `football-data` mediante fuzzy matching en tres rondas.

In [10]:
TEAM_MAP = build_team_mapping(df_xg, df_core, LEAGUE_MAP)

mapping_path = CONFIG_ROOT / "team_mapping_xg.json"
with open(mapping_path, "w") as f:
    json.dump(TEAM_MAP, f, indent=2, ensure_ascii=False)

print(f"Team mapping: {len(TEAM_MAP)} equipos → {mapping_path.relative_to(PROJECT_ROOT)}\n")
for us, core in list(sorted(TEAM_MAP.items()))[:5]:
    print(f"  {us:<20} →   {core:<20}")
print(f"  ...")

Team mapping: 34 equipos → config/team_mapping_xg.json

  Arminia Bielefeld    →   Bielefeld           
  Athletic Club        →   Ath Bilbao          
  Atletico Madrid      →   Ath Madrid          
  Bayer Leverkusen     →   Leverkusen          
  Borussia Dortmund    →   Dortmund            
  ...


### 5.2 Fechas

Tipo de dato y ejemplo en ambas fuentes.

In [11]:
print(f"  {'Fuente':<12} {'Tipo':<20} {'Ejemplo'}")
print(f"{'━'*50}")
print(f"  {'xG':<12} {str(df_xg['date'].dtype):<20} {df_xg['date'].iloc[0]}")
print(f"  {'core':<12} {str(df_core['Date'].dtype):<20} {df_core['Date'].iloc[0].strftime('%Y-%m-%d')}")

  Fuente       Tipo                 Ejemplo
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  xG           datetime64[ns]       2014-08-16 17:30:00
  core         datetime64[ns]       2014-08-16


#### 5.2.1 Desfase de fechas por huso horario


Identificación de partidos donde la fecha difiere entre Understat y football-data.

In [12]:
df_xg_flat = df_xg.reset_index()

k_xg = ("20" + df_xg_flat["season"].str[2:]) + "_" + \
       df_xg_flat["league"].map(LEAGUE_MAP) + "_" + \
       df_xg_flat["home_team"].map(TEAM_MAP).fillna(df_xg_flat["home_team"]) + "_" + \
       df_xg_flat["away_team"].map(TEAM_MAP).fillna(df_xg_flat["away_team"])

k_core = df_core["Season"].astype(str) + "_" + df_core["League"] + "_" + \
         df_core["HomeTeam"] + "_" + df_core["AwayTeam"]

df_test = pd.DataFrame({"xG": df_xg_flat["date"].dt.normalize().dt.date.values}, index=k_xg)
df_test = df_test.join(pd.Series(df_core["Date"].dt.date.values, index=k_core, name="core"), how="inner")
desfase = df_test[df_test["xG"] != df_test["core"]].dropna()

print(f"  Partidos con desfase de fecha: {len(desfase)} de {len(df_test):,}\n")
desfase_display = desfase.head(5).reset_index()
desfase_display.columns = ["Muestra de partidos inconsistentes", "xG", "core"]

display(desfase_display.style
    .hide(axis="index")
    .set_table_styles([
        {"selector": "th, td", "props": [("text-align", "center")]},
        {"selector": "th:first-child, td:first-child", "props": [("text-align", "left")]},
    ])
)

  Partidos con desfase de fecha: 95 de 10,660



Muestra de partidos inconsistentes,xG,core
2016_premier_Tottenham_Aston Villa,2015-11-03,2015-11-02
2016_premier_Crystal Palace_Sunderland,2015-11-24,2015-11-23
2016_premier_Everton_Crystal Palace,2015-12-08,2015-12-07
2016_premier_Leicester_Chelsea,2015-12-15,2015-12-14
2016_premier_Arsenal_Man City,2015-12-22,2015-12-21


> **Nota de implementación** — Inicialmente se consideró incluir la fecha como componente del `match_id` para garantizar unicidad. Sin embargo, el análisis revela que 95 partidos (0,9 % del dataset) presentan un desfase de ±1 día entre fuentes, atribuible a diferencias de huso horario en encuentros nocturnos — Understat registra en UTC, football-data en hora local. La clave `league + season + home_team + away_team` es igualmente unívoca dado que un equipo solo ejerce de local una vez frente a cada rival en la misma temporada, eliminando esta fuente de inconsistencia.

### 5.3 Ligas y temporadas

Nomenclatura de ligas y formato de temporadas.

In [13]:
leagues_xg   = sorted(df_xg.index.get_level_values("league").unique())
leagues_core = sorted(df_core["League"].unique())
seasons_xg   = sorted(df_xg.index.get_level_values("season").unique())
seasons_core = sorted(df_core["Season"].unique())

print(f"  Ligas xG          {leagues_xg}")
print(f"  Ligas core        {leagues_core}")
print()
print(f"  Temporadas xG     {seasons_xg}")
print(f"  Temporadas core   {seasons_core}")


  Ligas xG          ['ENG-Premier League', 'ESP-La Liga', 'GER-Bundesliga']
  Ligas core        ['bundesliga', 'laliga', 'premier']

  Temporadas xG     ['1415', '1516', '1617', '1718', '1819', '1920', '2021', '2122', '2223', '2324']
  Temporadas core   ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


---
##

## 6) Conclusiones del análisis exploratorio

### Cobertura

El dataset contiene **10.660 partidos × 17 variables**, con cobertura temporal idéntica al core clean:  
**380 partidos por temporada** en Premier League y La Liga y **306 en Bundesliga**, sin gaps en ninguna temporada.

### Integridad

No se detectaron **duplicados, nulos, valores negativos, drift de tipos ni outliers de xG**.  
Los goles (`home_goals`, `away_goals`) coinciden exactamente con el core dataset en las tres ligas.

### Variables útiles

Para el merge se incorporarán únicamente:

- **`home_xg`**
- **`away_xg`**

Las variables `home_goals` y `away_goals` se utilizan únicamente para validar el join.  
El resto corresponde a metadata de `Understat` y no se integrará en el dataset final.

### Compatibilidad con el core dataset

Se identifican algunas diferencias estructurales entre ambos datasets que deben resolverse antes del merge:

| Aspecto | Understat | Core clean | Acción |
|---|---|---|---|
| Equipos | Nombres completos | Algunos abreviados | Aplicar diccionario de mapping |
| Ligas | `ENG-Premier League` (ej.) | `premier` | Aplicar mapping |
| Temporadas | `1415` (ej.) | `2015` | Mapping en merge |
| `match_id` | — | `League_Season_Home_Away` | Construir tras aplicar mappings |

### Preparación del merge

El join se realizará mediante **`match_id`**, construido en el dataset xG tras aplicar los mappings de **equipos, ligas y temporadas**. Dado que la cobertura de partidos es idéntica al core dataset (**10.660 partidos**), se espera **coincidencia completa**. Cualquier discrepancia indicará que alguna variable no fue mapeada correctamente.

---
##

## 7) Exportación del raw xG dataset 

Guardado del dataset xG raw en Parquet para consumo en el notebook de merge.

In [14]:
RAW_XG_ROOT.mkdir(parents=True, exist_ok=True)
df_xg_export = df_xg.reset_index()

schema_xg = {
    "num_rows": len(df_xg_export),
    "num_columns": len(df_xg_export.columns),
    "columns": {col: str(df_xg_export[col].dtype) for col in df_xg_export.columns},
    "matches_per_league": df_xg_export.groupby("league").size().to_dict(),
    "seasons": sorted(df_xg_export["season"].unique().tolist()),
    "source": "understat (vía soccerdata)"
}

df_xg_export.to_parquet(XG_RAW_PATH, index=False)

with open(XG_VALIDATED_SCHEMA_PATH, "w") as f:
    json.dump(schema_xg, f, indent=2, ensure_ascii=False)

print(f"Dataset xG exportado:")
print(f"  {len(df_xg_export):,} filas × {len(df_xg_export.columns)} columnas\n")

print(f"Archivos guardados:")
print(f"  · Dataset → {XG_RAW_PATH.relative_to(PROJECT_ROOT)}")
print(f"  · Esquema → {XG_VALIDATED_SCHEMA_PATH.relative_to(PROJECT_ROOT)}")

Dataset xG exportado:
  10,660 filas × 20 columnas

Archivos guardados:
  · Dataset → data/raw/xg/xg_validated.parquet
  · Esquema → data/raw/xg/xg_validated_schema.json


---
##